# 🍾 Botellitas — Compilar APK Android

Ejecuta las celdas en orden. Tarda ~15 min la primera vez.

In [49]:
from google.colab import files
print('Sube botellitas_proyecto.zip')
uploaded = files.upload()
print('Subido:', list(uploaded.keys()))

Sube botellitas_proyecto.zip


Saving botellitas_proyecto.zip to botellitas_proyecto.zip
Subido: ['botellitas_proyecto.zip']


In [50]:
import os
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3-pip git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev cmake libffi-dev libssl-dev build-essential
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
!java -version
print('Entorno OK')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
Entorno OK


In [51]:
# Instalar buildozer SIN KivyMD
!pip install -q 'buildozer==1.5.0' 'cython==0.29.33' 'kivy==2.3.0' pillow
print('Buildozer instalado')

Buildozer instalado


In [52]:
import zipfile, shutil, os
if os.path.exists('/content/botellitas'): shutil.rmtree('/content/botellitas')
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name,'r') as z: z.extractall('/content/')
print('Extraído:', os.listdir('/content/botellitas'))

Extraído: ['windows_app', 'build_windows.py', 'android_app', 'core', '.gitignore', 'requirements_windows.txt', 'Compilar_APK_Colab.ipynb', 'README.md']


In [53]:
import shutil, os
BUILD_DIR = '/content/apk_build'
if os.path.exists(BUILD_DIR): shutil.rmtree(BUILD_DIR)
os.makedirs(BUILD_DIR)
for item in os.listdir('/content/botellitas/android_app'):
    src = f'/content/botellitas/android_app/{item}'
    dst = f'{BUILD_DIR}/{item}'
    if os.path.isdir(src): shutil.copytree(src, dst, dirs_exist_ok=True)
    else: shutil.copy2(src, dst)
shutil.copytree('/content/botellitas/core', f'{BUILD_DIR}/core', dirs_exist_ok=True)
print('Build dir:', sorted(os.listdir(BUILD_DIR)))

Build dir: ['buildozer.spec', 'core', 'main.py', 'views']


In [58]:
BUILD_DIR = '/content/apk_build'
spec = '''[app]
title = Botellitas
package.name = bot
source.include_exts = py,png,jpg,kellitas
package.domain = org.botellitas
source.dir = .
version = 1.2
requirements = python3,kivy==2.3.0,pillow,sqlite3
orientation = portrait
fullscreen = 0
android.permissions = READ_EXTERNAL_STORAGE,WRITE_EXTERNAL_STORAGE,MANAGE_EXTERNAL_STORAGE
android.api = 33
android.minapi = 21
android.ndk = 25b
android.archs = armeabi-v7a, arm64-v8a
android.allow_backup = True
android.manifest.legacy_external_storage = True
[buildozer]
log_level = 2
warn_on_root = 1
'''
with open(f'{BUILD_DIR}/buildozer.spec','w') as f: f.write(spec)
print('spec OK')

spec OK


In [60]:
# FIX sdkmanager — instalar cmdline-tools modernas
import os, subprocess, shutil, glob

BUILD_DIR = '/content/apk_build'
SDK_DIR = os.path.expanduser('~/.buildozer/android/platform/android-sdk')
TOOLS_DIR = f'{SDK_DIR}/cmdline-tools/latest'
os.makedirs(SDK_DIR, exist_ok=True)

if not os.path.exists(f'{TOOLS_DIR}/bin/sdkmanager'):
    print('Instalando cmdline-tools...')
    url = 'https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip'
    subprocess.run(['wget', '-q', url, '-O', '/tmp/cmdtools.zip'], check=True)
    subprocess.run(['unzip', '-q', '/tmp/cmdtools.zip', '-d', '/tmp/cmdtools'], check=True)
    os.makedirs(f'{SDK_DIR}/cmdline-tools', exist_ok=True)
    if os.path.exists(f'{SDK_DIR}/cmdline-tools/latest'):
        shutil.rmtree(f'{SDK_DIR}/cmdline-tools/latest')
    os.rename('/tmp/cmdtools/cmdline-tools', TOOLS_DIR)
    print('cmdline-tools instalado')
else:
    print('cmdline-tools ya instalado')

# Añadir al PATH
os.environ['PATH'] = f"{TOOLS_DIR}/bin:" + os.environ['PATH']
os.environ['ANDROID_SDK_ROOT'] = SDK_DIR

# Aceptar licencias
os.makedirs(f'{SDK_DIR}/licenses', exist_ok=True)
licencias = {
    'android-sdk-license': '\n8933bad161af4178b1185d1a37fbf41ea5269c55\nd56f5187479451eabf01fb78af6dfcb131a6481e\n24333f8a63b6825ea9c5514f83c2829b004d1fee',
    'android-sdk-preview-license': '\n84831b9409646a918e30573bab4c9c91346d8abd',
    'google-gdk-license': '\n33b6a2b64607f11b759f320ef9dff4ae5c47d97a',
    'intel-android-extra-license': '\nd975f751698a77b662f1254ddbeed3901e976f5a',
}
for nombre, contenido in licencias.items():
    with open(f'{SDK_DIR}/licenses/{nombre}','w') as f: f.write(contenido)
print('Licencias pre-aceptadas')

# --- NUEVA SECCIÓN: CREAR REDIRECCIÓN PARA BUILDOZER ---
# Esto evita el error "sdkmanager path ... does not exist"
OLD_TOOLS_BIN = f'{SDK_DIR}/tools/bin'
os.makedirs(OLD_TOOLS_BIN, exist_ok=True)
if not os.path.exists(f'{OLD_TOOLS_BIN}/sdkmanager'):
    # Creamos un enlace simbólico de la ruta vieja a la nueva
    os.symlink(f'{TOOLS_DIR}/bin/sdkmanager', f'{OLD_TOOLS_BIN}/sdkmanager')
    print('Redirección de sdkmanager creada (tools -> cmdline-tools)')
# -------------------------------------------------------

# Instalar build-tools via sdkmanager
result = subprocess.run(
    ['sdkmanager', '--sdk_root=' + SDK_DIR, 'build-tools;34.0.0', 'platform-tools', 'platforms;android-33'],
    input='y\ny\ny\n', text=True, capture_output=True
)
print('sdkmanager:', 'OK' if result.returncode==0 else result.stderr[-500:])

# Parchear buildozer para no pedir confirmación root
for bz_init in glob.glob('/usr/local/lib/python*/dist-packages/buildozer/__init__.py'):
    with open(bz_init,'r') as f: code = f.read()
    code = code.replace(
        "cont = input('Are you sure you want to continue [y/n]? ')",
        "cont = 'y'  # auto-aceptado"
    )
    with open(bz_init,'w') as f: f.write(code)
    print(f'buildozer parcheado: {bz_init}')

cmdline-tools ya instalado
Licencias pre-aceptadas
sdkmanager: OK
buildozer parcheado: /usr/local/lib/python3.12/dist-packages/buildozer/__init__.py


In [61]:
# COMPILAR
import subprocess, os, sys
BUILD_DIR = '/content/apk_build'
SDK_DIR   = os.path.expanduser('~/.buildozer/android/platform/android-sdk')
os.environ['ANDROID_SDK_ROOT'] = SDK_DIR
os.environ['PATH'] = f"{SDK_DIR}/cmdline-tools/latest/bin:{SDK_DIR}/platform-tools:" + os.environ['PATH']
os.chdir(BUILD_DIR)
print(f'Compilando desde: {os.getcwd()}')
print('Esto puede tardar 10-20 minutos...\n')
result = subprocess.run(
    ['buildozer', '-v', 'android', 'debug'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    input='y\n', text=True
)
with open('/content/buildozer_full.log','w') as f: f.write(result.stdout)
if result.returncode == 0:
    print('✅ APK compilado')
else:
    print('❌ ERROR')
    print('\n'.join(result.stdout.splitlines()[-80:]))

Compilando desde: /content/apk_build
Esto puede tardar 10-20 minutos...

✅ APK compilado


In [62]:
# Diagnóstico si hay error
with open('/content/buildozer_full.log') as f: lineas = f.readlines()
print(f'Total líneas: {len(lineas)}')
print('\n--- ERRORES ---')
for l in lineas:
    if any(k in l for k in ['ERROR','Error','Exception','FAILED','Traceback']): print(l.rstrip())
print('\n--- ÚLTIMAS 30 ---')
for l in lineas[-30:]: print(l.rstrip())

Total líneas: 294

--- ERRORES ---

--- ÚLTIMAS 30 ---
[DEBUG]:
[DEBUG]:   	You can use '--warning-mode all' to show the individual deprecation warnings and determine if they come from your own scripts or plugins.
[DEBUG]:
[DEBUG]:   	See https://docs.gradle.org/8.0.2/userguide/command_line_interface.html#sec:command_line_warnings
[DEBUG]:
[DEBUG]:   	BUILD SUCCESSFUL in 54s
[DEBUG]:   	33 actionable tasks: 32 executed, 1 up-to-date
[DEBUG]:   	
[DEBUG]:   	
[DEBUG]:   	
[DEBUG]:   	<-------------> 0% WAITING> IDLE> IDLE
[INFO]:    <- directory context /content/apk_build/.buildozer/android/platform/python-for-android
[DEBUG]:   All possible dists: [<Distribution: name bot with recipes (freetype, hostpython3, jpeg, libffi, openssl, png, sdl2_image, sdl2_mixer, sdl2_ttf, sqlite3, python3, sdl2, setuptools, pillow, six, pyjnius, android, kivy, urllib3, chardet, requests, certifi, idna)>]
[DEBUG]:   Dist matching name and arch: [<Distribution: name bot with recipes (freetype, hostpython3, 

In [63]:
import re

log_file_path = '/content/buildozer_full.log'
syntax_errors_by_file = {}

with open(log_file_path, 'r') as f:
    for line in f:
        # Look for lines containing 'SyntaxError' and attempting to extract file information
        match = re.search(r'File "(.*\.py)", line \d+, (SyntaxError:.*)', line)
        if match:
            filename = match.group(1)
            error_message = match.group(2)
            if filename not in syntax_errors_by_file:
                syntax_errors_by_file[filename] = []
            syntax_errors_by_file[filename].append(error_message.strip())

if syntax_errors_by_file:
    print('--- Syntax Errors found by file ---')
    for filename, errors in syntax_errors_by_file.items():
        print(f'\nArchivo: {filename}')
        # Print unique error messages to avoid repetition for the same file
        for error in sorted(list(set(errors))):
            print(f'  - {error}')
else:
    print('No se encontraron errores de tipo SyntaxError con información de archivo en el log.')


No se encontraron errores de tipo SyntaxError con información de archivo en el log.


In [64]:
import glob
from google.colab import files
BUILD_DIR = '/content/apk_build'
apks = glob.glob(f'{BUILD_DIR}/bin/*.apk')
if apks:
    print(f'APK: {apks[0]}')
    files.download(apks[0])
else:
    print('No se encontró APK. Ejecuta la celda de diagnóstico.')

APK: /content/apk_build/bin/bot-1.2-armeabi-v7a_arm64-v8a-debug.apk


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
# Descargar log completo
from google.colab import files
files.download('/content/buildozer_full.log')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>